# Offer extraction pipeline debug

This notebook mirrors `services.extract_offer.extract_offer` while keeping each stage visible. Use it to inspect and adjust split loading for long offers before touching the production pipeline.

In [23]:
from pathlib import Path
import importlib
import json
import os
import sys

ROOT = Path.cwd()
if not (ROOT / 'services').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

# Optional experiment knobs. Set before reloading services.extract_offer.
# Examples: EXPERIMENT_EXTRACT_MODE = 'one_shot', EXPERIMENT_EXTRACT_MODEL = 'gemini-3-pro-preview'
EXPERIMENT_EXTRACT_MODE = None
EXPERIMENT_EXTRACT_MODEL = None
if EXPERIMENT_EXTRACT_MODE:
    os.environ['EXTRACT_MODE'] = EXPERIMENT_EXTRACT_MODE
if EXPERIMENT_EXTRACT_MODEL:
    os.environ['EXTRACT_OFFER_MODEL'] = EXPERIMENT_EXTRACT_MODEL

from services.folder_handler import FolderHandler
import services.extract_offer as extract_offer

# Jupyter caches imported modules. Reload so notebook cells pick up edits in services/extract_offer.py.
extract_offer = importlib.reload(extract_offer)

CHUNKED_EXTRACTION_THRESHOLD = extract_offer.CHUNKED_EXTRACTION_THRESHOLD
POST_CHUNK_OVERLAP_LINES = extract_offer.POST_CHUNK_OVERLAP_LINES
POST_CHUNK_SIZE = extract_offer.POST_CHUNK_SIZE
EXTRACTION_MODE = extract_offer.EXTRACTION_MODE
EXTRACTION_MODEL_ID = extract_offer.EXTRACTION_MODEL_ID
are_duplicate_posts = extract_offer.are_duplicate_posts
extract_offer_chunked = extract_offer.extract_offer_chunked
extract_offer_one_shot = extract_offer.extract_offer_one_shot
filter_non_price_posts = extract_offer.filter_non_price_posts
format_chunked_llm_response = extract_offer.format_chunked_llm_response
merge_post_chunks = extract_offer.merge_post_chunks
normalize_amounts = extract_offer.normalize_amounts
parse_json_response = extract_offer.parse_json_response
parse_posts_response = extract_offer.parse_posts_response
post_identity = extract_offer.post_identity
read_pdf = extract_offer.read_pdf
read_txt = extract_offer.read_txt
split_text_chunks = extract_offer.split_text_chunks
validate_offer_json = extract_offer.validate_offer_json

STORAGE_DIR = ROOT / 'storage'
PROJECT_NAME = '40.40_pleisterwerk'
OFFER_NAME = 'bosma'
SAVE_RESULTS = False

folder_handler = FolderHandler(STORAGE_DIR)
offer_dir = STORAGE_DIR / PROJECT_NAME / OFFER_NAME
pdf_path = offer_dir / 'document.pdf'
raw_path = offer_dir / 'raw.txt'
extract_path = offer_dir / 'extract.json'

print(f'PDF: {pdf_path}')
print(f'Extraction mode: {EXTRACTION_MODE}')
print(f'Extraction model: {EXTRACTION_MODEL_ID}')
print(f'Existing raw text: {raw_path.exists()}')
print(f'Existing extract: {extract_path.exists()}')

PDF: /Users/timojolman/Zakelijk/UniPartners/vanWijnen/vanwijnen/development/app/storage/40.40_pleisterwerk/bosma/document.pdf
Existing raw text: True
Existing extract: True


## Load raw offer text

The production pipeline reads the PDF and then saves `raw.txt`. For debugging, this prefers the existing `raw.txt` so repeated runs are fast and deterministic.

In [24]:
if raw_path.exists():
    offer_text = raw_path.read_text()
else:
    offer_text = read_pdf(pdf_path)

lines = offer_text.splitlines()
mode = 'chunked' if len(offer_text) > CHUNKED_EXTRACTION_THRESHOLD else 'one_shot'
print(f'Characters: {len(offer_text)}')
print(f'Lines: {len(lines)}')
print(f'Threshold: {CHUNKED_EXTRACTION_THRESHOLD}')
print(f'Production mode: {mode}')
print('\nFirst 20 lines:')
for line_no, line in enumerate(lines[:20], start=1):
    print(f'{line_no:04d}: {line}')

Characters: 19034
Lines: 497
Threshold: 7000
Production mode: chunked

First 20 lines:
0001: info@bosmaplafonds.nl
0002: Bosma B.V.
0003: www.bosmaplafonds.nl
0004: De Boeg 9, 9206 BB Drachten
0005: Bank NL69 ABNA 04899290 52
0006: Tel. (0512) 51 59 85
0007: K.v.K. Leeuwarden nr. 01050968
0008: Fax (0512) 51 96 58
0009: BTW 65.95.315.B01
0010: Van Wijnen Gorredijk BV
0011: t.a.v. Riemer van Westervoort
0012: Badweg 42
0013: 8401 BL Gorredijk
0014: OF FERTE
0015: Datum 23­10­2025
0016: Of fertenummer O.2025­50452
0017: Ge ldigheid 28­10­2025
0018: Contactpersoon Simon Bosch
0019: Geachte Riemer van Westervoort
0020: Hierbij ontvangt u de offerte t.b.v. Nieuwb. IKC te St. Nicolaasga.


## Inspect split loading

Change `DEBUG_CHUNK_SIZE` and `DEBUG_OVERLAP_LINES` here while diagnosing the broken split behavior. The defaults match production.

In [25]:
DEBUG_CHUNK_SIZE = POST_CHUNK_SIZE
DEBUG_OVERLAP_LINES = POST_CHUNK_OVERLAP_LINES

chunks = split_text_chunks(
    offer_text,
    max_chars=DEBUG_CHUNK_SIZE,
    overlap_lines=DEBUG_OVERLAP_LINES,
)

print(f'Chunk size: {DEBUG_CHUNK_SIZE}')
print(f'Overlap lines: {DEBUG_OVERLAP_LINES}')
print(f'Chunks: {len(chunks)}')
for index, chunk in enumerate(chunks, start=1):
    chunk_lines = chunk.splitlines()
    print(
        f'{index:02d}: chars={len(chunk):5d}, lines={len(chunk_lines):4d}, '
        f'first={chunk_lines[0][:90] if chunk_lines else ""!r}, '
        f'last={chunk_lines[-1][:90] if chunk_lines else ""!r}'
    )

Chunk size: 4500
Overlap lines: 15
Chunks: 5
01: chars= 4440, lines= 138, first='info@bosmaplafonds.nl', last='SPARINGEN/ACHTERHOUT € 117,56'
02: chars= 4476, lines= 129, first='1x geperforeerd gips 12\xad20\xad35', last='stijlen hoh 600mm'
03: chars= 4488, lines= 114, first='BTW 65.95.315.B01', last='van aluminium U\u200b\xadprofielen v.z.v. zwart rubberprofiel. De verticale'
04: chars= 4491, lines= 111, first='Profielen in witte kleur', last='Inclusief, mits ononderbroken, uitgegaan van afhanghoogten welke met standaard'
05: chars= 3587, lines=  65, first='gewaarmerkte tekeningen, alsmede afdrukken van de E en W installatie tekeningen waarop de', last='Maatwerk in plafonds, wanden en kastwandsystemen'


In [26]:
def chunk_line_spans(text: str, chunks: list[str]) -> list[dict]:
    source_lines = text.splitlines()
    spans = []
    search_from = 0
    for index, chunk in enumerate(chunks, start=1):
        chunk_lines = chunk.splitlines()
        first = chunk_lines[0] if chunk_lines else ''
        start = None
        for candidate in range(search_from, len(source_lines)):
            if source_lines[candidate] == first:
                start = candidate
                break
        if start is None:
            start = search_from
        end = min(start + len(chunk_lines), len(source_lines))
        spans.append({
            'chunk': index,
            'start_line': start + 1,
            'end_line': end,
            'line_count': len(chunk_lines),
            'char_count': len(chunk),
        })
        search_from = max(end - DEBUG_OVERLAP_LINES, start + 1)
    return spans

spans = chunk_line_spans(offer_text, chunks)
spans

[{'chunk': 1,
  'start_line': 1,
  'end_line': 138,
  'line_count': 138,
  'char_count': 4440},
 {'chunk': 2,
  'start_line': 124,
  'end_line': 252,
  'line_count': 129,
  'char_count': 4476},
 {'chunk': 3,
  'start_line': 238,
  'end_line': 351,
  'line_count': 114,
  'char_count': 4488},
 {'chunk': 4,
  'start_line': 337,
  'end_line': 447,
  'line_count': 111,
  'char_count': 4491},
 {'chunk': 5,
  'start_line': 433,
  'end_line': 497,
  'line_count': 65,
  'char_count': 3587}]

In [27]:
CHUNK_INDEX = 1
CONTEXT_LINES = 25

span = spans[CHUNK_INDEX - 1]
start = max(span['start_line'] - CONTEXT_LINES, 1)
end = min(span['end_line'] + CONTEXT_LINES, len(lines))
print(f'Chunk {CHUNK_INDEX}: production span {span["start_line"]}-{span["end_line"]}; showing {start}-{end}')
for line_no in range(start, end + 1):
    marker = '>' if span['start_line'] <= line_no <= span['end_line'] else ' '
    print(f'{marker} {line_no:04d}: {lines[line_no - 1]}')

Chunk 1: production span 1-138; showing 1-163
> 0001: info@bosmaplafonds.nl
> 0002: Bosma B.V.
> 0003: www.bosmaplafonds.nl
> 0004: De Boeg 9, 9206 BB Drachten
> 0005: Bank NL69 ABNA 04899290 52
> 0006: Tel. (0512) 51 59 85
> 0007: K.v.K. Leeuwarden nr. 01050968
> 0008: Fax (0512) 51 96 58
> 0009: BTW 65.95.315.B01
> 0010: Van Wijnen Gorredijk BV
> 0011: t.a.v. Riemer van Westervoort
> 0012: Badweg 42
> 0013: 8401 BL Gorredijk
> 0014: OF FERTE
> 0015: Datum 23­10­2025
> 0016: Of fertenummer O.2025­50452
> 0017: Ge ldigheid 28­10­2025
> 0018: Contactpersoon Simon Bosch
> 0019: Geachte Riemer van Westervoort
> 0020: Hierbij ontvangt u de offerte t.b.v. Nieuwb. IKC te St. Nicolaasga.
> 0021: De offerte is opgesteld op basis van onderstaande stukken:
> 0022: Volgens de calculatietekeningen bijgevoegd als bijlage
> 0023: NIEUWB. IKC
> 0024: 1.57 1,00 m2 SYSTEEMPLAFONDS ALGEMEEN € 38,91 per m2 € 61.127,51
> 0025: Het leveren en monteren van een nieuw systeemplafond e.e.a. volgens
> 0026: ond

## Replay saved chunk responses

This mirrors the parsing/merge/filter/validate/normalize half of the production chunked pipeline using saved `llm_posts_chunk_*_response.txt` files.

In [28]:
summary_response_path = offer_dir / 'llm_summary_response.txt'
post_response_paths = sorted(offer_dir.glob('llm_posts_chunk_*_response.txt'))

print(f'Summary response: {summary_response_path.exists()}')
print(f'Post chunk responses: {len(post_response_paths)}')
for path in post_response_paths:
    print(path.name)

Summary response: True
Post chunk responses: 5
llm_posts_chunk_1_response.txt
llm_posts_chunk_2_response.txt
llm_posts_chunk_3_response.txt
llm_posts_chunk_4_response.txt
llm_posts_chunk_5_response.txt


## Show one chunk with its saved output

Change `CHUNK_WITH_OUTPUT` to inspect the exact text sent for a chunk together with the saved LLM response and parsed posts.

In [29]:
CHUNK_WITH_OUTPUT = 1

if CHUNK_WITH_OUTPUT < 1 or CHUNK_WITH_OUTPUT > len(chunks):
    raise ValueError(f'Choose a chunk between 1 and {len(chunks)}')

chunk = chunks[CHUNK_WITH_OUTPUT - 1]
span = spans[CHUNK_WITH_OUTPUT - 1]
response_path = offer_dir / f'llm_posts_chunk_{CHUNK_WITH_OUTPUT}_response.txt'

print('=' * 100)
print(f'CHUNK {CHUNK_WITH_OUTPUT}')
print(f'Lines: {span["start_line"]}-{span["end_line"]}')
print(f'Characters: {len(chunk)}')
print('=' * 100)
for offset, line in enumerate(chunk.splitlines(), start=span['start_line']):
    print(f'{offset:04d}: {line}')

print('\n' + '=' * 100)
print(f'SAVED LLM OUTPUT: {response_path.name}')
print('=' * 100)
if response_path.exists():
    raw_chunk_answer = response_path.read_text()
    print(raw_chunk_answer)

    posts, recovered = parse_posts_response(raw_chunk_answer)
    print('\n' + '=' * 100)
    print(f'PARSED POSTS: {len(posts)} recovered={recovered}')
    print('=' * 100)
    for post_index, post in enumerate(posts, start=1):
        print(f'\nPost {post_index}')
        print(json.dumps(post, ensure_ascii=False, indent=2))
else:
    print('No saved response found for this chunk. Run live extraction or pick a chunk with llm_posts_chunk_*_response.txt.')

CHUNK 1
Lines: 1-138
Characters: 4440
0001: info@bosmaplafonds.nl
0002: Bosma B.V.
0003: www.bosmaplafonds.nl
0004: De Boeg 9, 9206 BB Drachten
0005: Bank NL69 ABNA 04899290 52
0006: Tel. (0512) 51 59 85
0007: K.v.K. Leeuwarden nr. 01050968
0008: Fax (0512) 51 96 58
0009: BTW 65.95.315.B01
0010: Van Wijnen Gorredijk BV
0011: t.a.v. Riemer van Westervoort
0012: Badweg 42
0013: 8401 BL Gorredijk
0014: OF FERTE
0015: Datum 23­10­2025
0016: Of fertenummer O.2025­50452
0017: Ge ldigheid 28­10­2025
0018: Contactpersoon Simon Bosch
0019: Geachte Riemer van Westervoort
0020: Hierbij ontvangt u de offerte t.b.v. Nieuwb. IKC te St. Nicolaasga.
0021: De offerte is opgesteld op basis van onderstaande stukken:
0022: Volgens de calculatietekeningen bijgevoegd als bijlage
0023: NIEUWB. IKC
0024: 1.57 1,00 m2 SYSTEEMPLAFONDS ALGEMEEN € 38,91 per m2 € 61.127,51
0025: Het leveren en monteren van een nieuw systeemplafond e.e.a. volgens
0026: onderstaand specificaties.
0027: Type: Rockfon Blanka inleg
002

In [9]:
if summary_response_path.exists() and post_response_paths:
    summary_json = parse_json_response(summary_response_path.read_text())
    post_chunks = []
    recovered_chunks = []
    for index, path in enumerate(post_response_paths, start=1):
        posts, recovered = parse_posts_response(path.read_text())
        post_chunks.append(posts)
        if recovered:
            recovered_chunks.append(index)
        print(f'{path.name}: posts={len(posts)}, recovered={recovered}')

    merged_posts = merge_post_chunks(post_chunks)
    replayed_offer = {
        'Naam opdrachtgever': summary_json.get('Naam opdrachtgever'),
        'Totaalprijs inc. BTW': summary_json.get('Totaalprijs inc. BTW'),
        'Totaalprijs exc. BTW': summary_json.get('Totaalprijs exc. BTW'),
        'Posten': merged_posts,
    }
    replayed_offer['Posten'] = filter_non_price_posts(replayed_offer['Posten'])
    warnings = validate_offer_json(replayed_offer)
    replayed_offer = normalize_amounts(replayed_offer)
    print(f'Merged posts: {len(merged_posts)}')
    print(f'Final posts after filters: {len(replayed_offer["Posten"])}')
    print(f'Recovered chunks: {recovered_chunks}')
    print(f'Warnings: {warnings}')
else:
    post_chunks = []
    replayed_offer = None
    print('No saved chunk responses found for this offer.')

llm_posts_chunk_1_response.txt: posts=9, recovered=False
llm_posts_chunk_2_response.txt: posts=10, recovered=False
llm_posts_chunk_3_response.txt: posts=16, recovered=False
llm_posts_chunk_4_response.txt: posts=5, recovered=False
llm_posts_chunk_5_response.txt: posts=2, recovered=False
Merged posts: 42
Final posts after filters: 42
Recovered chunks: []
Warnings: ['Som van totaalposten (462308.49) komt niet overeen met totaal excl. BTW (449425.75)']


In [10]:
from collections import defaultdict

identity_locations = defaultdict(list)
flat_posts = []
for chunk_index, posts in enumerate(post_chunks, start=1):
    for post_index, post in enumerate(posts, start=1):
        location = (chunk_index, post_index, post)
        flat_posts.append(location)
        identity_locations[post_identity(post)].append(location)

exact_duplicates = {identity: locs for identity, locs in identity_locations.items() if len(locs) > 1}
fuzzy_duplicates = []
for left_index, (left_chunk, left_post_index, left_post) in enumerate(flat_posts):
    for right_chunk, right_post_index, right_post in flat_posts[left_index + 1:]:
        if left_chunk == right_chunk:
            continue
        if are_duplicate_posts(left_post, right_post):
            fuzzy_duplicates.append((left_chunk, left_post_index, left_post, right_chunk, right_post_index, right_post))

print(f'Exact duplicate identities: {len(exact_duplicates)}')
print(f'Fuzzy overlap duplicates: {len(fuzzy_duplicates)}')
for left_chunk, left_post_index, left_post, right_chunk, right_post_index, right_post in fuzzy_duplicates[:25]:
    print('\n')
    print(f'chunk={left_chunk}, post={left_post_index}: {left_post.get("Omschrijving", "")}')
    print(f'chunk={right_chunk}, post={right_post_index}: {right_post.get("Omschrijving", "")}')

Duplicate identities removed by merge_post_chunks: 0


## Optional live extraction

Set `RUN_LLM = True` in the setup cell to call Gemini. Set `SAVE_RESULTS = True` only when you want to overwrite the app's extraction artifacts for the selected offer.

In [ ]:
RUN_LLM = False
responses = {}
statuses = []

def capture_status(step, message=None, **kwargs):
    statuses.append({'step': step, 'message': message, **kwargs})
    print(step, '-', message)

def capture_response(name, answer):
    responses[name] = answer
    print(f'Captured {name}: {len(answer)} chars')

if RUN_LLM:
    prompt = read_txt(ROOT / 'prompts' / 'extract_prompt.txt')
    if len(offer_text) > CHUNKED_EXTRACTION_THRESHOLD:
        live_offer, live_answer = extract_offer_chunked(offer_text, capture_status, capture_response)
    else:
        live_offer, live_answer = extract_offer_one_shot(prompt, offer_text, capture_response)

    live_offer['Posten'] = filter_non_price_posts(live_offer.get('Posten', []))
    live_warnings = validate_offer_json(live_offer)
    live_offer = normalize_amounts(live_offer)
    print(f'Live posts: {len(live_offer.get("Posten", []))}')
    print(f'Live warnings: {live_warnings}')

    if SAVE_RESULTS:
        folder_handler.save_raw_pdf_text(pdf_path, offer_text)
        for name, answer in responses.items():
            if name == 'llm_response.txt':
                folder_handler.save_llm_response(pdf_path, answer)
            else:
                folder_handler.save_named_llm_response(pdf_path, name, answer)
        folder_handler.save_result(pdf_path, live_offer)
        print(f'Saved extract to {extract_path}')
else:
    print('RUN_LLM is False; skipped live extraction.')

RUN_LLM is False; skipped live extraction.
